In [25]:
import os
import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from torchvision import transforms
import torchvision.models as models
from torchvision.datasets import PCAM

import lightning as L
from lightning import LightningModule, LightningDataModule

In [26]:
EPOCHS = 30
BATCH_SIZE = 256
SEED = 42

In [27]:
# TODO: add good transforms for training and validation datasets (roto-translation, cropping, illuminance, etc.)
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Model

In [ ]:
class ResNet18Classifier(LightningModule):
    def __init__(self, num_classes: int = 2):
        super().__init__()
        self.model = models.resnet18(weights='DEFAULT')
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)
        self.num_classes = num_classes
        
        self.criterion = nn.CrossEntropyLoss()

        self.train_acc = torchmetrics.Accuracy(num_classes=num_classes)
        self.val_acc = torchmetrics.Accuracy(num_classes=num_classes)
        self.test_acc = torchmetrics.Accuracy(num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

    def _step(self, batch, stage: str):
        images, labels = batch

        outputs = self(images)
        loss = self.criterion(outputs, labels)
        preds = torch.argmax(outputs, dim=1)

        if stage == "train":
            self.train_acc(preds, labels)
        elif stage == "val":
            self.val_acc(preds, labels)
        elif stage == "test":
            self.test_acc(preds, labels)
        else:
            raise ValueError(f"Unknown stage: {stage}")
        
        self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"{stage}_acc", getattr(self, f"{stage}_acc"), on_step=False, on_epoch=True, prog_bar=True)
        
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, "train")
    def validation_step(self, batch, batch_idx):
        return self._step(batch, "val")
    def test_step(self, batch, batch_idx):
        return self._step(batch, "test")

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=1e-3, weight_decay=0.05)
        
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.1, patience=5
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1,
            },
        }

# Data Module

In [29]:
class PCAMDataModule(LightningDataModule):
    def __init__(self, data_root: str, name=""):
        super().__init__()
        self.train = PCAM(root=data_root, split='train', transform=train_transform, download=True)

        self.val = PCAM(root=data_root, split='val', transform=val_transform, download=True)

        self.test = PCAM(root=data_root, split='test', transform=val_transform, download=True)
        
        self.name = name

    def setup(self, stage: str = None):
        if stage == 'fit' or stage is None:
            self.train_dataset = self.train
            self.val_dataset = self.val
        if stage == 'test' or stage is None:
            self.test_dataset = self.test

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)

    def test_dataloader(self):
        if self.test_dataset is not None:
            return DataLoader(self.test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)
        else:
            return None

    def predict_dataloader(self):
        if self.test_dataset is not None:
            return DataLoader(self.test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)
        else:
            return None

In [30]:
model = ResNet18Classifier(num_classes=2)

In [31]:
model

ResNet18Classifier(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, 

In [32]:
datamodule = PCAMDataModule(data_root=os.environ.get("DATA_ROOT", "../data")) # Adjust the path as needed

In [33]:
ckbs = [
    L.pytorch.callbacks.ModelCheckpoint(
            monitor="val_loss",
            mode="min",
            save_top_k=1,
            filename="{epoch:02d}-{val_loss:.2f}",
        ),
    L.pytorch.callbacks.LearningRateMonitor(logging_interval='epoch'),
    #L.pytorch.callbacks.RichProgressBar(),
]

In [34]:
tensorboard_logger = L.pytorch.loggers.TensorBoardLogger(
    save_dir=os.environ.get("LOG_DIR", "../logs"),
)

In [35]:
trainer = L.Trainer(
    accelerator="auto", 
    devices="auto",
    max_epochs=EPOCHS,
    precision="16-mixed", # Use mixed precision for faster training
    num_nodes=1,
    logger=[tensorboard_logger],
    callbacks=ckbs,
    log_every_n_steps=10,
)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [36]:
#trainer.fit(model, datamodule=datamodule)